In [1]:
import requests
import lxml.html as lx
import re
import pandas as pd
import time

In [2]:
cat_url = 'https://cat.rescueme.org/California'
headers = {
    'User-Agent': "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:144.0) Gecko/20100101 Firefox/144.0"
}

In [3]:
url = 'https://post.rescueme.org/26-03-04-00099'
response = requests.get(url, headers = headers)
response.raise_for_status()
response.encoding = "utf-8"
test_html = lx.fromstring(response.text)

In [4]:
def get_pets_from_page(url):
    '''
    A function that extracts the links to all pet profiles located on a webpage.
    
    Arguments:
    url: URL of a web page
    
    Return:
    links: A list of all pet profile links found on the webpage
    '''
    try:
        result = requests.get(url, headers=headers)
        result.raise_for_status()
    except requests.exceptions.HTTPError:
        return None
    
    html = lx.fromstring(result.content)
    pet_links = html.xpath('//div[contains(@class, "card _cl _fa")]//a/@href')
    
    if not pet_links:
        return None
    
    links = []
    
    for href in pet_links:
        href = href.strip()
        if not href or href == "#":
            continue
        links.append(href)

    return links

In [5]:
def get_name(html):
    name = html.xpath('//span[contains(@class, "card-pet-name")]/text()')
    if name and name[0]:
        return name[0]
    return None

def get_breed(html):
    breed = html.xpath('//div[contains(@class, "summary-detail")]//span[contains(@class, "summary-heading")]/text()')
    if breed and breed[0]:
        return breed[0]
    return "Unknown"

def get_sex(html):
    sex = html.xpath('//div[contains(@class, "summary-detail")]//span[contains(@class, "summary-heading") and contains(text(), "Sex")]')
    if sex and sex[0].tail:
        return sex[0].tail.strip()
    return "Unknown"

def get_age(html):
    age = html.xpath('//div[contains(@class, "summary-detail")]//span[contains(@class, "summary-heading") and contains(text(), "Age")]')
    if age and age[0].tail:
        return age[0].tail.strip()
    return "Unknown"

def get_county(html):
    location = html.xpath('//div[contains(@class,"contact-content")]/p')
    if not location:
        return None
    lines = location[0].xpath('.//text()')
    lines = [line.strip() for line in lines if line.strip()]
    for line in lines:
        if "county" in line.lower():
            return line
    return None

def get_urgent(html):
    urgent = html.xpath('//div[contains(@class,"card-urgent")]/text()')
    name = get_name(html)
    if name is not None:
        name = name.split()
    else:
        name = []
    if urgent or any("urgent" in w.lower() for w in name):
        return "Yes"
    else:
        return "No"

def get_dog_compatibility(html):
    compatibility = html.xpath('//h3[contains(text(), "Compatibility")]/following-sibling::ul[contains(@class,"description-list")]/li/text()')
    compatibility = [i.strip().lower() for i in compatibility if i.strip()]
    for item in compatibility:
        if "dog" in item:
            if "good" in item:
                return "Good"
            if "not good" in item:
                return "Not Good"
    return "Unknown"

def get_cat_compatibility(html):
    compatibility = html.xpath('//h3[contains(text(), "Compatibility")]/following-sibling::ul[contains(@class,"description-list")]/li/text()')
    compatibility = [i.strip().lower() for i in compatibility if i.strip()]
    for item in compatibility:
        if "cat" in item:
            if "good" in item:
                return "Good"
            if "not good" in item:
                return "Not Good"
    return "Unknown"

def get_kid_compatibility(html):
    compatibility = html.xpath('//h3[contains(text(), "Compatibility")]/following-sibling::ul[contains(@class,"description-list")]/li/text()')
    compatibility = [i.strip().lower() for i in compatibility if i.strip()]
    for item in compatibility:
        if "not kid" in item:
                return "Not Good"
        if "kid" in item:
            if "good" in item:
                return "Good"
            if "not good" in item:
                return "Not Good"
    return "Unknown"

def get_energy_personality(html):
    personality = html.xpath('//h3[contains(text(), "Personality")]/following-sibling::ul[contains(@class,"description-list")]/li/text()')
    personality = [i.strip().lower() for i in personality if i.strip()]
    for item in personality:
        if "energy" in item:
            if "average" in item:
                return "Average"
            if "low" in item:
                return "Low"
            if "high" in item:
                return "High"
    return "Unknown"

def get_temperament_personality(html):
    personality = html.xpath('//h3[contains(text(), "Personality")]/following-sibling::ul[contains(@class,"description-list")]/li/text()')
    personality = [i.strip().lower() for i in personality if i.strip()]
    for item in personality:
        if "temperament" in item:
            if "average" in item:
                return "Average"
        if "dominant" in item:
            return "Dominant"
        if "submissive" in item:
                return "Submissive"
    return "Unknown"

def get_fixed_health(html):
    health = html.xpath('//h3[contains(text(), "Health")]/following-sibling::ul[contains(@class,"description-list")]/li/text()')
    health = [i.strip().lower() for i in health if i.strip()]
    for item in health:
        if "spay" in item or "neuter" in item:
            if "need" in item:
                return "No"
            else:
                return "Yes"
    return "Unknown"

def get_vaccine_health(html):
    health = html.xpath('//h3[contains(text(), "Health")]/following-sibling::ul[contains(@class,"description-list")]/li/text()')
    health = [i.strip().lower() for i in health if i.strip()]
    for item in health:
        if "vaccin" in item:
            if "need" in item:
                return "No"
            else:
                return "Yes"
    return "Unknown"

def get_description(html):
    description = html.xpath('//p[contains(@class, "animal-description")]/text()')
    if description and description[0]:
        return description[0]
    return None


In [6]:
def scrape_pets_from_urls(pet_urls, delay=1.0):
    all_pets = []

    for i, url in enumerate(pet_urls, 1):
        print(f"Scraping pet {i}/{len(pet_urls)}: {url}")
        try:
            r = requests.get(url, headers=headers)
            r.raise_for_status()
            html = lx.fromstring(r.text)

            pet_data = {
                "Name": get_name(html),
                "Breed": get_breed(html),
                "Sex": get_sex(html),
                "Age": get_age(html),
                "County": get_county(html),
                "Urgent Status": get_urgent(html),
                "Dog Compatibility": get_dog_compatibility(html),
                "Cat Compatibility": get_cat_compatibility(html),
                "Kid Compatibility": get_kid_compatibility(html),
                "Energy": get_energy_personality(html),
                "Temperament": get_temperament_personality(html),
                "Spayed/Neutered": get_fixed_health(html),
                "Vaccinated": get_vaccine_health(html),
                "Description": get_description(html),
                "URL": url
            }

            all_pets.append(pet_data)

        except Exception as e:
            print(f"Error scraping {url}: {e}")

        time.sleep(delay)

    df = pd.DataFrame(all_pets)
    return df

In [7]:
pet_urls = get_pets_from_page(cat_url)
df = scrape_pets_from_urls(pet_urls, delay=1.0)

Scraping pet 1/250: https://post.rescueme.org/26-03-04-00185
Scraping pet 2/250: https://post.rescueme.org/26-03-04-00102
Scraping pet 3/250: https://post.rescueme.org/26-03-04-00099
Scraping pet 4/250: https://post.rescueme.org/26-03-04-00097
Scraping pet 5/250: https://post.rescueme.org/26-03-04-00096
Scraping pet 6/250: https://post.rescueme.org/26-03-04-00094
Scraping pet 7/250: https://post.rescueme.org/26-03-04-00092
Scraping pet 8/250: https://post.rescueme.org/26-03-03-00288
Scraping pet 9/250: https://post.rescueme.org/26-03-02-00082
Scraping pet 10/250: https://post.rescueme.org/26-03-01-00313
Scraping pet 11/250: https://post.rescueme.org/26-03-01-00311
Scraping pet 12/250: https://post.rescueme.org/26-03-01-00293
Scraping pet 13/250: https://post.rescueme.org/26-02-28-00104
Scraping pet 14/250: https://post.rescueme.org/26-02-28-00100
Scraping pet 15/250: https://post.rescueme.org/26-02-27-00288
Scraping pet 16/250: https://post.rescueme.org/26-02-27-00257
Scraping pet 17/2

Scraping pet 133/250: https://post.rescueme.org/25-11-20-00327
Scraping pet 134/250: https://post.rescueme.org/25-11-18-00251
Scraping pet 135/250: https://post.rescueme.org/25-11-18-00168
Scraping pet 136/250: https://post.rescueme.org/25-11-10-00097
Scraping pet 137/250: https://post.rescueme.org/25-11-09-00255
Scraping pet 138/250: https://post.rescueme.org/25-11-09-00069
Scraping pet 139/250: https://post.rescueme.org/25-11-08-00148
Scraping pet 140/250: https://post.rescueme.org/25-11-08-00147
Scraping pet 141/250: https://post.rescueme.org/25-11-08-00128
Scraping pet 142/250: https://post.rescueme.org/25-11-08-00125
Scraping pet 143/250: https://post.rescueme.org/25-10-19-00084
Scraping pet 144/250: https://post.rescueme.org/25-09-29-00295
Scraping pet 145/250: https://post.rescueme.org/25-09-29-00293
Scraping pet 146/250: https://post.rescueme.org/25-09-28-00289
Scraping pet 147/250: https://post.rescueme.org/25-09-28-00286
Scraping pet 148/250: https://post.rescueme.org/25-09-1

In [9]:
df.to_csv("cats_california.csv", index=False)